# Classifier v4: does the citing paragraph help?

Every model so far has read only the citing sentence. Here BiomedBERT (3 labels, sqrt class
weights, 4 epochs, our reranker's evidence) reads one of three versions of the citation:

- **none**: the citing sentence alone, as before;
- **window**: the citing sentence with the sentence before and after it;
- **paragraph**: the whole citing paragraph (up to 1500 characters around the citation).

Built by `scripts/build_context.py` from the citing paragraph, located by the citation marker;
the annotators' marked context span is not used, since a real user would not have it.

**Phase 1 picks the variant by 3-fold cross-validation over the 2141 training citations**, cut by
article, instead of on dev (only 24 NOT ENOUGH INFO citations), whose choices did not transfer to
test before. **Phase 2** trains the winning context variant on all of train with 3 seeds and
predicts test.

**Before running:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all, and allow
Google Drive access. About **2 to 2.5 hours**. Each finished run is saved to
`MyDrive/CiteAudit_runs/v4`; if Colab disconnects, Run all again and finished runs are skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
RES = '/content/drive/MyDrive/CiteAudit_runs/v4'
import os; os.makedirs(RES, exist_ok=True)

%cd /content
!rm -rf CiteAudit && git clone -q https://github.com/SapnilPatel/CiteAudit.git
%cd /content/CiteAudit
!git log -1 --oneline
!pip -q install "transformers>=4.44" "tokenizers>=0.19"

import subprocess, json, statistics
def run(cmd):
    noise = ('UNEXPECTED', 'MISSING', 'LOAD REPORT', 'Key ', '----', 'Notes', '- UNEXP', '- MISS', 'Loading weights')
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if not any(n in line for n in noise):
            print(line, end='')
    if p.wait():
        raise RuntimeError(f'failed ({p.returncode}): {cmd}')

T = 'data_train/target'
MAX_CLAIM = {'none': 128, 'window': 192, 'paragraph': 320}  # room for the first segment, in tokens
def train(name, context, seed, extra):
    out = f'{RES}/{name}'
    if os.path.exists(f'{out}/config.json'):
        print(f'{name}: already done'); return
    run(f'python scripts/train_classifier.py --train {T}/ours_20_train_cv --dev {T}/ours_20 '
        f'--context {context} --max-claim {MAX_CLAIM[context]} --labels 3 --class-weight sqrt '
        f'--seed {seed} --amp --eval-batch 16 --out checkpoints/{name} --pred-dir {out}.part {extra}')
    run(f'cp checkpoints/{name}/config.json {out}.part/ && rm -rf checkpoints/{name}')
    os.rename(f'{out}.part', out)

LABELS = ['ACCURATE', 'NOT_ACCURATE', 'NOT ENOUGH INFO']
def scores(rows):
    f1 = []
    for c in LABELS:
        tp = sum(r['gold'] == c and r['prediction'] == c for r in rows)
        fp = sum(r['gold'] != c and r['prediction'] == c for r in rows)
        fn = sum(r['gold'] == c and r['prediction'] != c for r in rows)
        f1.append(2 * tp / (2 * tp + fp + fn) if tp else 0.)
    return [sum(r['gold'] == r['prediction'] for r in rows) / len(rows), sum(f1) / 3] + f1
read = lambda p: [json.loads(l) for l in open(p)]

## Phase 1: 3-fold cross-validation over train (about 1.5 hours)

In [ ]:
VARIANTS = ['none', 'window', 'paragraph']
for k in range(3):
    for v in VARIANTS:
        train(f'cv_{v}_f{k}', v, 0, f'--folds 3 --fold {k}')

print(f'{"context":10s} {"n":>5s} ' + ' '.join(f'{c:>9s}' for c in ['accuracy', 'macro-F1', 'ACC F1', 'N_ACC F1', 'NEI F1']))
cv = {}
for v in VARIANTS:
    rows = sum((read(f'{RES}/cv_{v}_f{k}/oof.jsonl') for k in range(3)), [])
    cv[v] = scores(rows)
    print(f'{v:10s} {len(rows):5d} ' + ' '.join(f'{x:9.3f}' for x in cv[v]))
BEST = max(['window', 'paragraph'], key=lambda v: cv[v][1])
print(f'\nbest context variant by cross-validated macro-F1: {BEST} '
      f'({cv[BEST][1]:.3f} against {cv["none"][1]:.3f} without context)')
json.dump({'cv': cv, 'best': BEST}, open(f'{RES}/cv_summary.json', 'w'), indent=2)

## Phase 2: the winning variant on all of train, 3 seeds (about 40 to 60 minutes)

In [ ]:
for seed in (0, 1, 2):
    train(f'final_{BEST}_s{seed}', BEST, seed, f'--predict test={T}/ours_20_test')

runs = [scores(read(f'{RES}/final_{BEST}_s{s}/ours_20_test.jsonl')) for s in (0, 1, 2)]
print(f'test, {BEST} context, mean ± sd over 3 seeds')
for name, xs in zip(['accuracy', 'macro-F1', 'ACC F1', 'N_ACC F1', 'NEI F1'], zip(*runs)):
    print(f'  {name:9s} {statistics.mean(xs):.3f} ± {statistics.stdev(xs):.3f}')
print('without context (classifier_v3.md, E3): accuracy 0.662, macro-F1 0.489')

## Download

Zips `MyDrive/CiteAudit_runs/v4` (predictions and logs, no weights). Put the unzipped `v4` folder
in the repository root and tell Claude.

In [ ]:
!cd /content/drive/MyDrive/CiteAudit_runs && zip -qr /content/v4.zip v4 -x '*.part/*'
from google.colab import files
files.download('/content/v4.zip')